# Vontex — first model training notebook
Run the cells from top to bottom. This trains a very small Transformer from randomly initialized weights. It is for learning, not a capable assistant. Replace the sample training text with your own or public-domain/openly licensed text. Do not upload private data or copyrighted material without permission.

In [ ]:
# Vontex — first tiny language model
# This is an educational experiment, not a ChatGPT-level assistant.
import os, time, torch
print("PyTorch:", torch.__version__)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Training device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU assigned. You can still test the notebook, but training will be slower.")

In [ ]:
# TRAINING TEXT
# Replace the sample below with text you wrote yourself or material that is
# public-domain / openly licensed and permitted for model training.
# Do not paste private information or copyrighted books/articles without permission.
TRAIN_TEXT = """
Vontex is a small experimental language model created as a learning project.
A language model learns patterns from examples of text. It receives a sequence
of characters and tries to predict what character comes next.
Training adjusts the model's numerical parameters so that its predictions
become closer to the examples. A small model has limited memory and knowledge.
It may produce repetitive, incomplete, or incorrect sentences.
The purpose of this first experiment is to understand tokenization, neural
networks, training loss, and text generation.
A computer represents text as numbers. This notebook uses UTF-8 bytes as tokens.
The model reads a short context and predicts the next byte in the sequence.
"""
print("Training characters:", len(TRAIN_TEXT))
print("Tip: more varied, permitted training text generally gives better learning.")

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F

# UTF-8 byte tokenizer: each token is an integer from 0 to 255.
data = torch.tensor(list(TRAIN_TEXT.encode("utf-8")), dtype=torch.long)
vocab_size = 256
block_size = 128
if len(data) <= block_size + 1:
    raise ValueError("Please add more training text (at least 130 UTF-8 bytes).")

def get_batch(batch_size=16):
    starts = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in starts])
    y = torch.stack([data[i+1:i+block_size+1] for i in starts])
    return x.to(device), y.to(device)

class VontexGPT(nn.Module):
    def __init__(self):
        super().__init__()
        width = 128
        self.token_emb = nn.Embedding(vocab_size, width)
        self.pos_emb = nn.Embedding(block_size, width)
        layer = nn.TransformerEncoderLayer(
            d_model=width, nhead=4, dim_feedforward=512,
            dropout=0.1, batch_first=True, norm_first=True
        )
        self.blocks = nn.TransformerEncoder(layer, num_layers=2)
        self.norm = nn.LayerNorm(width)
        self.head = nn.Linear(width, vocab_size)

    def forward(self, idx, targets=None):
        b, t = idx.shape
        positions = torch.arange(t, device=idx.device)
        x = self.token_emb(idx) + self.pos_emb(positions)[None, :, :]
        # True values mask future positions.
        mask = torch.triu(torch.ones(t, t, device=idx.device, dtype=torch.bool), diagonal=1)
        x = self.blocks(x, mask=mask)
        logits = self.head(self.norm(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, vocab_size), targets.reshape(-1))
        return logits, loss

model = VontexGPT().to(device)
print("Parameters:", sum(p.numel() for p in model.parameters()))
print("Model initialized with random weights.")

In [ ]:
# Train the model. On Kaggle, turn on a GPU in Notebook settings if one is available.
# Training time depends on the assigned hardware and the size of your text.
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
steps = 1000
batch_size = 16
start = time.time()
model.train()

for step in range(steps):
    xb, yb = get_batch(batch_size)
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    if (step + 1) % 100 == 0:
        print(f"Step {step+1}/{steps} | loss: {loss.item():.4f} | elapsed: {(time.time()-start)/60:.1f} min")

os.makedirs("/kaggle/working", exist_ok=True)
checkpoint_path = "/kaggle/working/vontex_tiny.pt"
torch.save({"model_state_dict": model.state_dict()}, checkpoint_path)
print("Checkpoint saved:", checkpoint_path)

In [ ]:
# Generate a short sample from the trained model.
@torch.no_grad()
def generate(prompt="Vontex", max_new_tokens=250, temperature=0.8):
    model.eval()
    ids = torch.tensor([list(prompt.encode("utf-8"))], dtype=torch.long, device=device)
    for _ in range(max_new_tokens):
        context = ids[:, -block_size:]
        logits, _ = model(context)
        next_logits = logits[:, -1, :] / max(temperature, 0.1)
        probs = F.softmax(next_logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        ids = torch.cat([ids, next_id], dim=1)
    return bytes(ids[0].tolist()).decode("utf-8", errors="replace")

print(generate())
print("\nSaved file:", checkpoint_path)
print("Use the notebook's Output panel to download vontex_tiny.pt after saving a version.")
